# Bài học 08 - Mẫu Thiết Kế Đa Tác Nhân


## Cài đặt


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Tại sao lại dùng Hệ thống Đa tác nhân?

Các nhiệm vụ trong thế giới thực như lập kế hoạch chuyến đi liên quan đến nhiều loại chuyên môn khác nhau — logistics, kiến thức địa phương, ngân sách và nhiều hơn nữa. Một tác nhân duy nhất cố gắng xử lý mọi thứ nhanh chóng trở nên khó quản lý.

Hệ thống đa tác nhân giải quyết vấn đề này thông qua **chuyên môn hóa**: mỗi tác nhân tập trung vào một lĩnh vực chuyên môn, tạo ra kết quả chất lượng cao hơn so với một người tổng quát. Chúng cũng cải thiện **khả năng mở rộng** — bạn có thể thêm các tác nhân mới (ví dụ: chuyên gia chuyến bay, nhà phê bình nhà hàng) mà không cần viết lại quy trình làm việc hiện có. Các tác nhân hợp tác với nhau thông qua một quy trình có cấu trúc, truyền bối cảnh từ người này sang người tiếp theo.


## Tạo các Đại lý Chuyên biệt


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Xây dựng một quy trình tuần tự

`WorkflowBuilder` cho phép bạn kết nối các tác nhân thành một đồ thị có hướng. Ở đây chúng ta tạo một chuỗi hai bước đơn giản: **TravelPlanner** soạn thảo lịch trình, sau đó **TravelConcierge** xem xét và cải tiến nó.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Thêm Nhiều Tác Nhân Hơn Vào Quy Trình Làm Việc

Một trong những lợi thế lớn nhất của mẫu đa tác nhân là sự dễ dàng trong việc mở rộng. Dưới đây chúng tôi thêm một tác nhân **BudgetReviewer** để kiểm tra kế hoạch so với ngân sách của người đi du lịch, đánh dấu những mục có thể làm vượt quá chi phí và đề xuất các lựa chọn tiết kiệm tiền. Quy trình làm việc giờ đây chạy ba tác nhân theo trình tự:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Tóm tắt

Trong bài học này, bạn đã học cách:

1. **Tạo các đại lý chuyên biệt** — mỗi đại lý có vai trò tập trung (lập kế hoạch, lễ tân, xem xét ngân sách).
2. **Kết nối các đại lý vào một quy trình làm việc tuần tự** sử dụng `WorkflowBuilder` và `add_edge`.
3. **Phát dòng đầu ra** từ một chuỗi đa đại lý, theo dõi đại lý nào đang nói.
4. **Mở rộng quy trình làm việc** bằng cách thêm các đại lý mới vào chuỗi mà không sửa đổi các đại lý hiện có.

Mô hình thiết kế đa đại lý giữ cho mỗi đại lý đơn giản trong khi tạo ra kết quả phong phú và được xem xét kỹ lưỡng hơn bất kỳ đại lý đơn lẻ nào có thể đạt được một mình.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố từ chối trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể có lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ bản địa nên được coi là nguồn chính xác và đáng tin cậy. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm về bất kỳ sự hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
